# <center> Erweiterte Ausführungen zu Beispiel 5.2 <center>
## <center> Auswertung experimenteller Daten: Anpassung kinetischer Parameter <center>
Bearbeitet von Sina Ramsayer.

Dieses Beispiel ergänzt den im Lehrbuch (Beispiel 5.2) beschriebenen graphischen Lösungsansatz zur Bestimmung kinetischer Parameter. Die zugrunde liegenden experimentellen Daten, die vereinfachte chemische Reaktion $\mathrm{A_1} \rightarrow \mathrm{A_2}$ sowie der als bekannt angenommene kinetische Potenzansatz:

$$
    r = k \ c_1^{n_1}
$$

werden dabei unverändert übernommen. Wie üblich wird die Reaktionsgeschwindigkeitskonstante $k$ über den Arrhenius-Ansatz beschrieben:

$$
    k = k_0 \ exp\left({\frac{-E_\mathrm{A}} {RT}}\right).
$$

Die Bestimmung der Parameter $k_0, E_\mathrm{A}$ und $n_1$ erfolgt nun mithilfe moderner numerischer Methoden, wie sie bereits im Git-Repository zu Beispiel 5.2 eingeführt werden.

Im Folgenden liegt der Fokus daher darauf, an einem einfachen und anschaulichen Beispiel zu zeigen, mit welchen grundlegenden Strategien die Güte, Stabilität und Konvergenz numerischer Lösungsansätze verbessert werden können. Diese Methoden ermöglichen es, auch mehrdimensionale und deutlich komplexere Parameterräume, wie sie bei realen Reaktionsnetzwerken vorliegen, robust anzupassen.

Die zuverlässige Identifikation kinetischer Parameter stellt dabei auch in der aktuellen Forschung eine zentrale Herausforderung dar. Insbesondere bei hochdimensionalen Reaktionsnetzwerken und transienten Betriebsbedingungen ist die Parameter-Findbarkeit häufig limitiert. Stagge und Güttel [1] adressieren diese Problematik im Kontext heterogen katalysierter Reaktionssysteme mithilfe physikalisch strukturierter neuronaler Netzwerkmodelle (hCRNNs) und verdeutlichen damit die anhaltende Bedeutung robuster Parameteridentifikation.

In [ ]:
import numpy as  np
import pandas as pd
import scipy
import scipy.integrate as integ
from lmfit import Minimizer, Parameters, report_fit
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

Zunächst werden die vorliegenden experimentellen Daten eingelesen und zur Veranschaulichung graphisch dargestellt. Der entsprechende Plot zeigt die Verläufe der Konzentration $c$ über die Zeit $t$ für verschiedene Temperaturen $T$.

In [ ]:
T = np.array([80, 65, 50]) + 273.15 # temperature in K
T_ref = np.mean(T) # reference temperature in K
print('T_ref:', T_ref)

# load data
file = "data/example_kinetics_data1_1.txt"
data = np.genfromtxt(file, names=True)

t = np.array([data["t"], data["t"], data["t"]]) # time in h
c = np.array([data["T80"], data["T65"], data["T50"]]) # concentration in mol m-3

# define consistent colors and markers for each experiment
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(T)))
markers = ["o", "^", "d"]

# visualize concentration profiles
fig, ax = plt.subplots()

for i in range(len(T)):
    ax.scatter(t[i], c[i], color=colors[i], marker=markers[i], label=f"$T$ = {T[i]-273.15:.0f} °C")

ax.set_xlabel(r"$t$ / h")
ax.set_ylabel(r"$c$ / mol $\mathrm{m^{-3}}$")
ax.grid(True, linestyle='--', alpha=0.5)
ax.tick_params(direction='in')
ax.legend()
plt.show()

Die Bestimmung kinetischer Parameter wie der präexponentielle Faktor $k_0$ und der Aktivierungsenergie $E_\mathrm{A}$ ist zentral, um Reaktionsgeschwindigkeiten zuverlässig zu beschreiben. Verschiedene äquivalente Umformulierungen der Arrhenius-Gleichung beeinflussen dabei die numerische Stabilität, die Parameterkorrelation und das Konvergenzverhalten im Anpassungs-Prozess.
Der klassische Arrhenius-Ansatz (wie bereits oben eingeführt) lautet:

$$
k(T) = k_0 \exp\left(-\frac{E_\mathrm{A}}{R T}\right),
$$

wobei $R$ die universelle Gaskonstante und $T$ die absolute Temperatur ist. Diese Form eignet sich direkt für die nicht lineare Parameteranpassung und ist physikalisch besonders anschaulich. Jedoch können die Parameter $k_0$ und $E_\mathrm{A}$ nicht unabhängig voneinander bestimmt werden, da sie stark miteinander gekoppelt sind [1]. Vor allem bei ungünstig gewählten Startwerten und großen Temperaturbereichen kann dies zu langsamer Konvergenz oder der Bestimmung fehlerhafter Parameterkombinationen führen.

Zur Verbesserung der Stabilität und geringeren Kopplung der Parameter wird oft eine dezentrierte (decoupled) Form der Arrheniusgleichung verwendet. Ausgangspunkt ist die Definition einer Referenztemperatur $T_\mathrm{ref}$ mit $k_\mathrm{ref} = k(T_\mathrm{ref})$:

$$
k_\mathrm{ref} = k_0 \exp\left(-\frac{E_\mathrm{A}}{R T_\mathrm{ref}}\right).
$$

Wird die Gleichung umgeformt und anschließend $k_0 = k_\mathrm{ref} \exp\left(\frac{E_\mathrm{A}}{R T_\mathrm{ref}}\right)$ in die ursprüngliche Gleichung eingesetzt, ergibt sich die dezentralisierte Form:

$$
k(T) = k_\mathrm{ref} \exp\left(\frac{E_\mathrm{A}}{R T_\mathrm{ref}}\right)
\exp\left(-\frac{E_\mathrm{A}}{R T}\right).
$$

 Vereinfacht lautet die Gleichung folgendermaßen:

$$
k(T) = k_\mathrm{ref} \exp\Big[-\frac{E_\mathrm{A}}{R} \Big(\frac{1}{T} - \frac{1}{T_\mathrm{ref}}\Big)\Big].
$$

Durch Bezug der Reaktionsgeschwindigkeit auf einen Referenzwert $k_\mathrm{ref}$, wird die Kovarianz zwischen $k_0$ und $E_\mathrm{A}$ bei der Anpassung reduziert, da die Exponentialfunktion um $T_\mathrm{ref}$ entlastet wird.

Zur Verbesserung der Leistungsfähigkeit und Stabilität eines Optimierungsverfahrens kann es zudem sinnvoll sein, die zu optimierenden Parameter in den logarithmierten Raum zu transformieren. Dies ist vor allem sinnvoll, wenn sich die möglichen Parameterbereiche über mehrere Größenordnungen erstrecken. Durch die Transformation kann der Algorithmus den gesamten Parameterbereich besser erfassen und die Wahrscheinlichkeit eine global gute Lösung der Optimierung zu finden steigt. Im Falle des Arrhenius-Ansatzes bietet sich die Linearisierung der Gleichung an, anstelle von $k_0$ wird nun $\mathrm{ln}(k_0)$ numerisch bestimmt.

$$
\ln k(T) = \ln k_0 - \frac{E_\mathrm{A}}{R T}, \quad
\ln k(T) = \ln k_\mathrm{ref} - \frac{E_\mathrm{A}}{R} \Big(\frac{1}{T} - \frac{1}{T_\mathrm{ref}}\Big)
$$

Bei Verwendung der linearisierten Gleichung ist eine sorgfältige Rücktransformation der optimierten Parameter notwendig, um physikalisch sinnvolle Ergebnisse zu gewährleisten.

Je nach Datenbereich, Rauschlevel und Startwerten der Parameter, kann die Arbeit mit den verschiedenen Gleichungen den Anpassungsprozess stabilisieren und die Konvergenz verbessern. In der nachfolgenden Methode `compute_k` können durch Übergabe des Attributs `method` die zur Parameteranpassung herangezogene Form der Arrhenius-Gleichung, und damit die zu optimierenden Parameter, festgelegt werden.


In [ ]:
def compute_k(T, params, method, T_ref):
    """
    Compute the reaction rate constant k(T) using different Arrhenius-based formulations.

    Parameters
    ----------
    T : array_like
        Temperature(s) in K at which the rate constant should be evaluated.
    params : lmfit.Parameters
        Parameter set containing the required kinetic parameters.
        The required keys depend on the chosen method.
    method : {'arrhenius', 'decoupled', 'log', 'decoupled_log'}
        Selects the formulation used to compute k.
    T_ref : float
        Reference temperature in K used for the decoupled Arrhenius formulations.

    Returns
    -------
    float or ndarray
        The reaction rate constant(s) evaluated at temperature T.
    """

    R = scipy.constants.R

    if method == "arrhenius":
        k0 = params["k0"].value
        E_A = params["E_A"].value
        return k0 * np.exp(-E_A / (R * T))

    elif method == "decoupled":
        k_ref = params["k_ref"].value
        E_A = params["E_A"].value
        return k_ref * np.exp(-E_A / R * (1/T - 1/T_ref))

    elif method == "log":
        ln_k0 = params["ln_k0"].value
        E_A = params["E_A"].value
        ln_k = ln_k0 - E_A/(R*T)
        return np.exp(ln_k)

    elif method == "decoupled_log":
        ln_k_ref = params["ln_k_ref"].value
        E_A = params["E_A"].value

        ln_k = ln_k_ref - (E_A / R * (1/T - 1/T_ref))
        return np.exp(ln_k)

    else:
        raise ValueError("Unknown method for Arrhenius model")


Die Methode `ode` implementiert das zugrunde liegende kinetische Potenzgesetz.

In [ ]:
def ode(t, c, n_1, k):
    """
    Calculate time derivatives of the ode for a batch CSTR.
    Parameters
    ----------
    t: float
        time of the calculation in h
    c: array
        concentration values at time t in mol m-3
    n_1: float
        reaction order
    k: float
        reaction rate constant


    Returns
    -------
    dcdt: array
        time derivatives of the concentration
    """

    # calculating the derivatives
    dcdt = -k * c**n_1 # d[A]/dt
    return dcdt

Die Funktion `residual` gibt die Abweichung zwischen den experimentell gemessenen Konzentrationsverläufen und den durch das kinetische Modell simulierten Konzentrationen zurück. Dafür wird zuerst die Reaktionsgeschwindigkeitskonstante mithilfe der Funktion `compute_k` entsprechend der gewählten Arrhenius-Form berechnet. Anschließend wird das Anfangswertproblem der kinetischen Differentialgleichung, die in `ode` definiert ist, numerisch gelöst. Hierzu wird numpys `solve_ivp` Funktion verwendet. Als Abweichung zwischen simulierten und experimentellen Ergebnissen wird in diesem Fall die Differenz an allen Messzeitpunkten und für alle Temparatuen bestimmt. Diese Differenz soll minimiert werden, woraus sich das Optimierungsproblem ergibt. Für bestimmte Anwendungsszenarien kann es sinnvoll sein, verschiedene Datensätze oder den Beginn bzw. das Ende eines Zeitintervalls mehr oder weniger bei der Fehlerberechnung zu gewichten. Das kann zu einer Lösung der Optimierung führen, die die gewünschten Bereiche besonders gut abbilden kann.

In [ ]:
def residual(params, times, data, T, T_ref, method="decoupled"):
    """
    Residual function for lmfit. Calculate the difference between simulation and experiment for provided data.

    Parameters
    ----------
    params: Parameters
        Parameter object containing all variables to be estimated [n_1, k_0, E_A]
    times: list
        List of arrays with times in h of sampling for each experiment
    data: list
        List of arrays with concentration values in mol m-3 from experiments
    T: list
        List of temperatures for each experiment.
    T_ref: float
        Reference temperature in K.
    method: {'arrhenius', 'decoupled', 'log', 'decoupled_log'}
        Method used to fit params.
    Returns
    -------
    residuals: array
        flattened array of differences between simulation and experimental data.
    """
    n_1 = params['n_1'].value

    residuals = []

    for i in range(len(T)):
        t_i = times[i]
        c_init = data[i][0]
        # select model
        k_i = compute_k(T[i], params, method, T_ref)

        sol = integ.solve_ivp(
            ode,
            [t_i[0], t_i[-1]],
            [c_init],
            method='LSODA',
            t_eval=t_i,
            rtol=1e-9,
            atol=1e-12,
            dense_output=False,
            args=(n_1, k_i)
        )
        residuals.extend(sol.y[0] - data[i])

    return np.array(residuals)

Da sich abhängig von der gewählten Arrhenius-Form die tatsächlich optimierten Parameter unterscheiden, dient die Funktion `get_fitted_params` dazu, die Ergebnisse der Optimierung in eine einheitliche und physikalisch interpretierbare Form zu überführen. Unabhängig von der verwendeten Anpassungsmethode werden die kinetischen Parameter ausgelesen und auf die Größen $k_0$, $E_\mathrm{A}$ und $n_1$ zurückgeführt.

In [ ]:
def get_fitted_params(res):
    """
    Extraction of fitted parameters from MinimizerResult.

    Parameters
    ----------
    res: MinimizerResult

    Returns
    -------
    list
        Fitted parameters [k_0, E_A, n_1]
    """

    R = scipy.constants.R
    if "k0" in res.params:
        return [res.params["k0"].value, res.params["E_A"].value, res.params["n_1"].value]
    elif "k_ref" in res.params:
        E_A = res.params["E_A"].value
        k0 = res.params["k_ref"].value * np.exp(E_A / (R * T_ref))
        return [k0, E_A, res.params["n_1"].value ]
    elif "ln_k0" in res.params:
        E_A = res.params["E_A"].value
        k0 = np.exp(res.params["ln_k0"].value)
        return [k0, E_A, res.params["n_1"].value ]
    elif "ln_k_ref" in res.params:
        E_A = res.params["E_A"].value
        k0 = np.exp(res.params["ln_k_ref"].value) * np.exp(E_A / (R * T_ref))
        return [k0, E_A, res.params["n_1"].value ]
    else:
        raise ValueError("Unknown method for fitting parameters")


Im Folgenden werden die kinetischen Parameter numerisch und simultan bestimmt. In diesem Beispiel wird dazu das Python-Paket `lmfit` verwendet, welches die zu optimierenden Parameter explizit verwaltet und automatisch statistische Kenngrößen wie Parameterunsicherheiten und Korrelationen berechnet. Die zu schätzenden Modellparameter werden in `lmfit` mithilfe eines `Parameters`-Objekts definiert. Jeder Parameter wird dabei durch einen Namen, einen Startwert sowie optionale Unter- und Obergrenzen des Parameterbereichs beschrieben. Die Wahl geeigneter Startwerte und sinnvoller Parametergrenzen ist entscheidend, um eine stabile und effiziente Konvergenz des Optimierungsverfahrens zu gewährleisten und eine globale Lösung zu erhalten. In unserem Fall sind die Startwerte und Parametergrenzen so gewählt, dass sie physikalisch plausible Bereiche abdecken, jedoch keine zusätzlichen Annahmen über die exakten Parameterwerte erzwingen.

Die eigentliche Optimierung erfolgt über ein `Minimizer`-Objekt. Dieses verbindet, die zuvor definierte `residual`-Funktion, den Satz der zu optimierenden Parameter, sowie zusätzliche Argumente wie experimentelle Daten oder Modelloptionen. Durch Aufruf dessen Funktion `minimize` wird die Optimierung ausgeführt. Die `residual`-Funktion liefert für einen gegebenen Parametersatz die Abweichungen zwischen Modellvorhersage und Messdaten. Der Optimierer passt die Parameter iterativ so an, dass eine Zielfunktion, per default die Summe der quadrierten Residuen ("leastsq"), minimiert wird. Da die Residuen hier aus der numerischen Lösung eines Anfangswertproblems resultieren, muss dafür in jedem Optimierungsschritt ein vollständiger ODE-solver aufgerufen werden.

Neben dem Standardverfahren (Levenberg-Marquardt) unterstützt `lmfit` eine Vielzahl weiterer Algorithmen, darunter gradientenfreie lokale Verfahren (z.B. Nelder-Mead), globale Optimierer (z.B. Differential Evolution) oder stochastische Verfahren (z.B. Simulated Annealing) [1]. Je nach Komplexität des Parameterraums können bestimmte Algorithmen besser geeignet sein als andere. Häufig kann auch eine zweistufige Strategie bei der Parameterbestimmung helfen: zunächst ein globales Verfahren zur groben Lokalisierung des Extremums, gefolgt von einem lokalen Algorithmus zur Feinoptimierung. Durch gezielte Wahl des Anpassungsalgorithmus und Modifikation weiterer zugehöriger Parameter kann für komplexe Reaktionsnetzwerke eine bessere Lösung des Optimierungsproblems gefunden werden.

Weitere Informationen zur `minimize`-Funktion und möglichen zusätzlichen Attributen sind unter folgendem Link: https://lmfit.github.io/lmfit-py/fitting.html#lmfit.minimizer.Minimizer, zusammengetragen [2].

In [ ]:
# initilize Parameters object
params1 = Parameters()
params1.add("n_1", value=1.0, min=0, max=3)
params1.add("k0", value=1e11, min=0, max=1e13)
params1.add("E_A", value=80000, min=1e3, max=1e6)

# define Minimizer object
min1 = Minimizer(residual, params1,
                 fcn_args=(t, c, T, T_ref, "arrhenius"))
res1 = min1.minimize() # perform optimization
k0_1, E_A_1, n_1 = get_fitted_params(res1) # extract fitted parameters

params2 = Parameters()
params2.add("n_1", value=1.0, min=0, max=3)
params2.add("k_ref", value=1, min=1e-2, max=10)
params2.add("E_A", value=80000, min=1e3, max=1e6)

min2 = Minimizer(residual, params2,
                 fcn_args=(t, c, T, T_ref, "decoupled"))
res2 = min2.minimize()
k0_2, E_A_2, n_2 = get_fitted_params(res2)

params3 = Parameters()
params3.add("n_1", value=1.0, min=0, max=3)
params3.add("ln_k_ref", value=-0.01, min=-10, max=3)            # log-preexponential
params3.add("E_A", value=80000, min=1e3, max=1e6)

min3 = Minimizer(residual, params3,
                 fcn_args=(t, c, T, T_ref, "decoupled_log"))
res3 = min3.minimize()
k0_3, E_A_3, n_3 = get_fitted_params(res3)


Nun werden die Ergebnisse der drei Anpassungsstrategien gegenübergestellt. Obwohl alle Ansätze für dieses sehr simple Beispiel zu nahezu identischen optimalen Parametern (siehe geringe Parameterunsicherheit) und Anpassungsgüten (siehe chi-square Werte nahe null) führen, unterscheiden sie sich im numerischen Verhalten des Optimierungsverfahrens.

Der klassische Arrhenius-Ansatz erfordert eine deutlich höhere Anzahl an Funktionsauswertungen (=101) im Vergleich zu der dezentrierten und logarithmierten Form (=17). Bei Integration der kinetischen Berechnungen in Reaktormodellierungen und komplexeren kinetischen Modellen hat dies einen entscheidenden Einfluss auf die Laufzeit der Simulation. Außerdem zeigt der Anpassungsbericht des klassischen Ansatzes eine höhere Korrelation zwischen dem Stoßfaktor $k_0$ und der Aktivierungsenergie $E_\mathrm{A}$ von +0.9244 im Vergleich zu den beiden anderen Ansätzen mit C($k_\mathrm{ref}$, $E_\mathrm{A}$) = -0.5776.


In [ ]:
print("\n=== Classical Arrhenius ===")
report_fit(res1)

print("\n=== Decoupled Arrhenius ===")
report_fit(res2)

print("\n=== Logarithmic Decoupled Arrhenius ===")
report_fit(res3)


Die von den Algorithmen bestimmten Parameter werden nochmals in der untenstehenden Tabelle miteinander verglichen. Während die Wahl der Arrhenius Form in diesen Fall für das Endergebnis keine große Rolle spielt, können jedoch für reale Netzwerke die Güte der Lösung und die Laufzeit deutlich beeinträchtigt werden.

In [ ]:
# create a dictionary for the comparison
data_compare = {
    "Model": ["Arrhenius", "Decoupled", "Decoupled (log)"],
    "E_A": [E_A_1, E_A_2, E_A_3],
    "n_1": [n_1, n_2, n_3],
    "k0": [k0_1, k0_2, k0_3],
    "Chi-square": [res1.chisqr, res2.chisqr, res3.chisqr],
    "Reduced Chi-square": [res1.redchi, res2.redchi, res3.redchi],
}
df_compare = pd.DataFrame(data_compare)

# display the DataFrame
df_compare


Zuletzt können die mit den numerisch bestimmten Parametern berechneten und die experimentellen Konzentrationen vergleichend gegen die Zeit aufgetragen werden. Da die Werte sich kaum unterscheiden werden im unteren Plot beispielhaft die Ergebnisse des klassischen Arrhenius-Ansatzes dargestellt. Es ist zu erkennen, dass die experimentellen Datenpunkte jeweils auf den simulierten Kurven liegen, was die Güte des Modells und der angepassten Parameter bestätigt.

In [ ]:
# create figure and axis
fig, ax = plt.subplots(figsize=(7, 5))

for i, T_i in enumerate(T):
    # solve ODE for this experiment
    k_i = compute_k(T_i, res1.params, 'arrhenius', T_ref)
    sol = integ.solve_ivp(
        ode,
        [t[i][0], t[i][-1]],
        [c[i][0]],
        t_eval=t[i],
        args=(n_1, k_i),
    )

    # plot experimental data
    ax.plot(
        t[i],
        c[i],
        linestyle='',
        marker=markers[i],
        color=colors[i],
        # label=f'Exp {i+1} data ({T_i-273:.0f} K)' # custom legend
    )

    # plot simulation
    ax.plot(
        t[i],
        sol.y[0],
        linestyle='-',
        color=colors[i],
        # label=f'Exp {i+1} sim' # custom legend
    )

# -----------------------
# Custom legend
# Legend 1: Colors = Temperatures
temp_handles = [
    Line2D([0], [0], color=colors[i], marker='s', markersize=10, linestyle='', label=f'{T[i]:.0f} K')
    for i in range(len(T))
]
temp_labels = [f'{T[i]-273:.0f} °C' for i in range(len(T))]

# Legend 2: Markers = Experimental data
exp_tuple = tuple(
    Line2D([], [], linestyle='', marker=m, color='black')
    for m in markers
)

# Legend 3: Line = Simulation
sim_handle = Line2D([0], [0], color='black', lw=2, label='Sim')

# Build combined list of handles
combined_handles = (
    temp_handles                 # all temperature color entries
    + [exp_tuple]               # one entry for experimental markers
    + [sim_handle]              # one entry for simulation line
)
# Corresponding labels
combined_labels = (
    temp_labels
    + ["Exp"]
    + ["Sim"]
)
# Create final legend
ax.legend(
    handles=combined_handles,
    labels=combined_labels,
    handler_map={tuple: HandlerTuple(ndivide=None)},
    loc="upper right",
    frameon=True,
    ncol=2,
)
# -----------------------

# Formatting
ax.set_xlabel(r'$t$ / h', fontsize=12)
ax.set_ylabel(r"$c$ / mol $\mathrm{m^{-3}}$", fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)
ax.tick_params(direction='in')
ax.set_title('Experimental Data vs Model Simulation', fontsize=13)

plt.tight_layout()
plt.show()

Dieses Beispiel zeigt, wie klassische kinetische Modelle mithilfe numerischer ODE-Solver und moderner Optimierungsverfahren robust an experimentelle Daten angepasst werden können. Dabei wird deutlich, dass die Wahl einer geeigneten Parametrisierung, (z.B. durch dezentrale oder logarithmierte Arrhenius-Formen) einen entscheidenden Einfluss auf Konvergenz, Stabilität und Interpretierbarkeit der Anpassungsergebnisse hat. Über klassische Optimierungsverfahren hinaus eröffnen datengetriebene Modellansätze und Methoden des maschinellen Lernens vielversprechende Perspektiven für die kinetische Modellierung. Eine vertiefende Diskussion zur Parameter-Findbarkeit in komplexen, heterogen katalysierten Reaktionsnetzwerken unter Verwendung physikalisch interpretierbarer neuronaler Netzwerkmodelle findet sich bei Stagge und Güttel [1].

Der vorgestellte Code ist modular aufgebaut und lässt sich ohne grundlegende Änderungen auf komplexere Reaktionssysteme erweitern, beispielsweise auf mehrstufige Reaktionen, Reaktionsnetzwerke oder zusätzliche experimentelle Freiheitsgrade. Damit bietet das Beispiel eine flexible Grundlage zum Experimentieren und zur schrittweisen Übertragung der gezeigten Methoden auf realistische kinetische Fragestellungen.

## Literaturverzeichnis

[1] H. Stagge, R. Güttel, The findability of microkinetic parameters by heterogeneous chemical reaction neural networks (hCRNNs), Chemical Engineering Journal 510 (2025) 161460. https://doi.org/10.1016/j.cej.2025.161460. \
[2] Performing Fits and Analyzing Outputs — Non-Linear Least-Squares Minimization and Curve-Fitting for Python, (n.d.). https://lmfit.github.io/lmfit-py/fitting.html#lmfit.minimizer.Minimizer (accessed February 6, 2026).



